In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
.master("local[4]") \
.appName("DataCleaning") \
.config("spark.executor.memory","4g") \
.config("spark.driver.memory","2g") \
.getOrCreate()

In [ ]:
adult_train_df = spark.read \
.option("header","True") \
.option("inferSchema","True") \
.option("sep",",") \
.csv("adult.data") 

In [ ]:
adult_test_df = spark.read \
.option("header","True") \
.option("inferSchema","True") \
.option("sep",",") \
.csv("adult.test") 

In [ ]:
adult_whole_df = adult_train_df.union(adult_test_df)
adult_whole_df.limit(5).toPandas().head()

 # 1. s?tunlara bo?luk kontrol?

In [ ]:
from pyspark.sql.functions import *

In [ ]:
adult_whole_df1 = adult_whole_df \
.withColumn("workclass", trim(col("workclass"))) \
.withColumn("education", trim(col("education"))) \
.withColumn("marital_status", trim(col("marital_status"))) \
.withColumn("occupation", trim(col("occupation"))) \
.withColumn("relationship", trim(col("relationship"))) \
.withColumn("race", trim(col("race"))) \
.withColumn("sex", trim(col("sex"))) \
.withColumn("native_country", trim(col("native_country"))) \
.withColumn("output", trim(col("output"))) 

In [ ]:
adult_whole_df1.count()

#  OUTPUT ???NDEK? "." TEM?ZL???

In [ ]:
adult_whole_df2 = adult_whole_df1 \
.withColumn("output", regexp_replace(col("output"), "<=50K.","<=50K")) \
.withColumn("output", regexp_replace(col("output"), ">50K.",">50K"))

In [ ]:
adult_whole_df2.groupBy(col("output")).agg({"*":"count"}) \
.toPandas().head()

# 3. NULL KONTROL?


In [ ]:
sayac_for_null = 1
for sutun in adult_whole_df2.columns:
    if(adult_whole_df2.filter(col(sutun).isNull()).count() > 0 ):
        print(sayac_for_null, ". ", sutun, " i?inde null var.")
    else:
        print(sayac_for_null, sutun)
    sayac_for_null += 1

# 4. "?" KONTROLLER?

In [ ]:
sayac_for_question = 1
for sutun in adult_whole_df2.columns:
    if(adult_whole_df2.filter(col(sutun).contains("?")).count() > 0 ):
        print(sayac_for_question, ". ", sutun, " i?inde ? var.")
    else:
        print(sayac_for_question, sutun)
    sayac_for_question += 1

In [ ]:
adult_whole_df2.select("workclass","occupation","native_country","output") \
.filter(
    col("workclass").contains("?") |
    col("occupation").contains("?") |
    col("native_country").contains("?")
    
) \
.groupBy("workclass","occupation","native_country","output").count() \
.orderBy(col("count").desc()) \
.toPandas().head(25)

In [ ]:
adult_whole_df3 = adult_whole_df2 \
.filter(~(
    col("workclass").contains("?") |
    col("occupation").contains("?") |
    col("native_country").contains("?")
))

print(adult_whole_df2.count())
print(adult_whole_df3.count())

# 5. ZAYIF SINIFLARIN KALDIRILMASI

In [ ]:
adult_whole_df4 = adult_whole_df3 \
.filter(~(
    col("workclass").contains("never-worked") | col("workclass").contains("without-pay") | 
       col("occupation").contains("Armed-Forces") | col("native_country").contains("Holand-Netherlands")
))

print(adult_whole_df3.count())
print(adult_whole_df4.count())

# 6. E??T?M DURUMUYLA ?LG?L? KATEGOR?LER? B?RLE?T?RME

In [ ]:
adult_whole_df5 = adult_whole_df4.withColumn("education_merged",
when(col("education").isin("1st-4th", "5th-6th", "7th-8th"), "Elementary-School") 
.when(col("education").isin("9th", "10th", "11th", "12th"), "High-School") 
.when(col("education").isin("Masters","Doctorate"), "Postgraduate")
.when(col("education").isin("Bachelors","Some-college"), "Undergraduate")
.otherwise(col("education"))
                                            
)

adult_whole_df5.select("education","education_merged").toPandas().head(20)

In [ ]:
nitelik_siralama = ["workclass", "education", "education_merged", "marital_status", "occupation", "relationship", "race",
     "sex", "native_country", "age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week","output"]

adult_whole_df6 = adult_whole_df5.select(nitelik_siralama)
adult_whole_df6.toPandas().head()

In [ ]:
df =adult_whole_df6

In [ ]:
df =df.toPandas()

In [ ]:
df.to_csv('Cleaned_adult', sep=',', encoding='utf-8')